In [ ]:
import os
import re
import numpy as np
from datetime import datetime, timedelta
import html
import urllib
from bs4 import BeautifulSoup

In [ ]:
def fix_encoding(text):
    """
    - Encodes the html entities in a text into UTF-8 encoding. For instance, "I&amp;m ..." => "I'm ..."
    - Case folding
    :param text: string.
    :return: fixed text(sting)
    """
    a = html.unescape(text) # html to utf-8
    return a

In [ ]:
def replace_month(text):
    dict_months = {
        'janvier':'january',
        'février':'february',
        'mars':'march',
        'avril':'april',
        'mai':'may',
        'juin':'june',
        'juillet':'july',
        'août':'august',
        'septembre':'september',
        'octobre':'october',
        'novembre':'november',
        'décembre':'december'
    }
    for month in dict_months.keys():
        if month in text:
            return text.replace(month, dict_months[month])

In [ ]:
def extract_data_from_page(page, method='path'):
    
    # Load page
    if method=='path':
        with open(pagepath, encoding='utf8') as fp: #################### add encoding
            soup = BeautifulSoup(fp, 'html.parser')
    else:
        soup = BeautifulSoup(page, 'html.parser')
    # Initialize output
    data = {
        "title":None,
        "poster":None,
        "date":None,
        "duration":None,
        "genres":None,
        "directors":[],
        "actors":[],
        "notes":{'press':None, 'spectators':None},
        "synopsis":None
           }
    
    # Title
    tag_title = soup.find(attrs={"class":"titlebar-title-lg"})
    data["title"] = tag_title.getText()
    
    # Poster
    tag_poster = soup.find(attrs={"class":"thumbnail-img"})
    data["poster"] = tag_poster.attrs["src"]
    
    # Meta Infos
    tag_info = soup.find(attrs={"class":"meta-body-info"})
    p = re.compile('\S')
    infos = "".join(p.findall(tag_info.get_text()))
    date, duration, genres = infos.split('/')
    data['date'] = datetime.strptime(replace_month(date), '%d%B%Y') # Date
    data['duration'] = datetime.strptime(duration, '%Hh%Mmin') - datetime.strptime('0','%H') # Duration
    data['genres'] = genres.split(',') # Genres
    
    # Directors
    tag_dirs = soup.find(attrs={"class":"meta-body-direction"})
    directors = tag_dirs.find_all(attrs={"class":"blue-link"})
    data["directors"] = [director.getText() for director in directors]
    
    # Actors
    tag_act = soup.find(attrs={"class":"meta-body-actor"})
    data["actors"] = [child.string for child in list(tag_act.children) if '\n' not in child.string][1:]
    
    # Notes
    tag_notes = soup.find_all(attrs={"class":"stareval-note"})
    data["notes"]["press"] = float(tag_notes[0].getText().replace(',', '.'))
    data["notes"]["spectators"] = float(tag_notes[1].getText().replace(',', '.'))
    
    # Synopsis
    tag_syn = soup.find(attrs={"id":"synopsis-details"})
    tag_syn = tag_syn.find(attrs={"class":"content-txt"})
    p = re.compile('[\s]{2,}')
    text = tag_syn.getText()
    for space in p.findall(text):
        text = text.replace(space, '')
    data["synopsis"] = text
    
    return data

In [ ]:
# pagepath='../html/Joker - film 2019 - AlloCiné.html'
# pagepath='../html/The Dark Knight, Le Chevalier Noir - film 2008 - AlloCiné.html'
# pagepath='../html/The Big Lebowski - film 1998 - AlloCiné.html'
url = "http://www.allocine.fr/film/fichefilm_gen_cfilm=258374.html"
page = urllib.request.urlopen(url).read()
extract_data_from_page(page, method='url')

___